# Do it yourself: Bayesian yes–no decisions with binary data

So far, you've been handed sliders on a machine someone else built. This one hands you the parts. You write the lines of code at the heart of the inference — the visualizations are already wired to whatever you return.

The object is a test or classifier with **known sensitivity and specificity**, operating under an **unknown base rate (prevalence)**. We want to infer the **PPV** (and, by the same logic, the NPV), which depend on that unknown.

We are going to build the apparatus of Bayesian inference and decision step by step.

## 1 · The data model

- We assume that sensitivity and specificity are known and fixed
- The base rate / prevalence is unknown
- We want to infer the PPV, the positive negative predictive value (same as precision)
- As data, we have a certain number of samples that were predicted as positive or negative. Only the predicted condition is known, we don't know the true labels

Also: $$PPV = \frac{\mathrm{sensitivity} \cdot \mathrm{prevalence}}{\mathrm{sensitivity} \cdot \mathrm{prevalence} + (1 - \mathrm{specificity}) \cdot (1 - \mathrm{prevalence})}$$


In [ ]:
from functools import partial
from typing import Callable

import numpy as np
from scipy import stats, integrate
import matplotlib.pyplot as plt

### 1.1 · The generative model and the likelihood

We reason in terms of a process that could generate the data we observe — that leads us to the right form for the likelihood. The guiding question is: *"if I knew the prevalence, how likely would the predicted positive and negative counts I have be?"*

We assume the data are i.i.d.: the generative model draws each datum independently from the same distribution. So we can focus on a single datum. What's the chance for one datum to be labeled positive?

To be labeled positive, a datum has to be truly positive **and** correctly labeled, **or** truly negative **and** mislabeled. Call that whole "probability of being predicted positive" $\theta$:

$$\theta =  \mathrm{sensitivity} \cdot \mathrm{prevalence} + (1- \mathrm{specificity}) \cdot (1 - \mathrm{prevalence})$$

Because sensitivity and specificity are known, knowing $\theta$ is the same as knowing the prevalence:

In [ ]:
def prevalence_from_theta(theta: float, *, sensitivity: float, specificity: float) -> float:
    return (theta + specificity - 1) / (sensitivity + specificity - 1)

⚠️ The formula above breaks down when $\mathrm{sensitivity} + \mathrm{specificity} = 1$ (division by zero). Can you see why that makes sense? *(Hint: substitute $\mathrm{specificity} = 1 - \mathrm{sensitivity}$ into the expression for $\theta$ above — what does $\theta$ depend on then?)*

How are the labels we observe generated given a value for $\theta$? Each sample is a Bernoulli trial with success probability $\theta$. So, the probability of observing a certain count of positive and negative labels is given by the [binomial distribution](https://en.wikipedia.org/wiki/Binomial_distribution), implemented in [scipy.stats.binom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binom.html).

In [ ]:
def likelihood(theta: float, *, num_positive_labels: int, num_negative_labels: int) -> float:
    num_samples = num_positive_labels + num_negative_labels
    return stats.binom.pmf(num_positive_labels, num_samples, theta)

## 2 · The prior

The parameter $\theta$ is a proportion, thus contained in the interval $[0, 1]$. You can pick any prior that reflects how much you know about theta, e.g., an uniform prior. 

You should know that a binomial likelihood with a fixed number of trials has a [conjugate prior](https://en.wikipedia.org/wiki/Conjugate_prior), which makes computing the posterior trivial (and is also easy to interpret, its strength being measured as a number of pseudo-observations). The conjugate prior for the binomial is the [beta distribution](https://en.wikipedia.org/wiki/Beta_distribution), implemented in in [scipy.stats.beta](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.beta.html).

(Incidentally: the uniform distribution for in the interval $[0, 1]$ is also a beta distribution!)

In [ ]:
def prior(theta: float, *, alpha: float = 1.0, beta: float = 1.0) -> float:
    # A Beta(alpha, beta) density. alpha = beta = 1 is the uniform prior;
    # (alpha - 1) and (beta - 1) act as pseudo-counts of prior positives and negatives.
    return stats.beta.pdf(theta, alpha, beta)

## 3 · Computing the unnormalized posterior

Bayes' theorem says the posterior is proportional to prior times likelihood:

$$p(\theta \mid \text{data}) \;\propto\; p(\theta)\,\cdot\,p(\text{data}\mid\theta)$$

Up to the constant that makes it integrate to one, that's the whole story — a **pointwise multiplication** of the two functions you just wrote. No grid, no loop: we keep it as a function of $\theta$ and multiply.

In [ ]:
def get_unnormalized_posterior(
        prior: Callable[[float], float],
        likelihood: Callable[[float], float]
) -> Callable[[float], float]:
    return lambda theta: prior(theta) * likelihood(theta)

## 4 · Normalizing the posterior

The missing constant is $\int_0^1 p(\theta)\,p(\text{data}\mid\theta)\,\mathrm{d}\theta$ — the **evidence**. It rescales the curve so it integrates to one.

This step, apparently so simple, is *the* central challenge of Bayesian inference: in real models that integral is high-dimensional and has no closed form, which is the entire reason MCMC, variational inference, and their friends exist. Here $\theta$ is one-dimensional on $[0, 1]$, so plain numerical integration does the job.

(You don't always need the posterior normalized — to compare the *odds* of two hypotheses the constant cancels — but to read off probabilities and expectations, you do.)

In [ ]:
def normalize_posterior(raw_posterior: Callable[[float], float]) -> Callable[[float], float]:
    # Integrate the unnormalized posterior to get the denominator of Bayes' theorem,
    # the evidence. theta lives on [0, 1], so we integrate over that interval.
    evidence, _ = integrate.quad(raw_posterior, 0.0, 1.0)
    return lambda theta: raw_posterior(theta) / evidence

## 5 · From $\theta$ to the quantity we care about

We now have the posterior for $\theta$, but we set out to learn the **PPV** — the probability that a *positive verdict* is actually correct. Luckily the PPV is a deterministic function of $\theta$: each $\theta$ fixes one prevalence, and each prevalence one PPV (and vice-versa). In fact, since $\theta$ is exactly $P(\text{predicted } +)$,

$$\mathrm{PPV} \;=\; \frac{\mathrm{sensitivity}\cdot\mathrm{prevalence}}{\theta}.$$

One technicality, easy to miss: a posterior is a *density*, not a value, so we can't just relabel the x-axis. Where the map from $\theta$ to PPV stretches the axis, the density has to compress (and vice-versa) so that total probability stays at one — that local stretch factor is the **Jacobian** $\left|\mathrm{d}\theta/\mathrm{d}\,\mathrm{PPV}\right|$. We estimate it numerically below, so you never have to differentiate anything by hand.

In [ ]:
def ppv_from_theta(theta: float, *, sensitivity: float, specificity: float) -> float:
    prevalence = prevalence_from_theta(theta, sensitivity=sensitivity, specificity=specificity)
    return sensitivity * prevalence / theta


def theta_from_ppv(ppv: float, *, sensitivity: float, specificity: float) -> float:
    youden = sensitivity + specificity - 1
    return sensitivity * (specificity - 1) / (ppv * youden - sensitivity)


def change_of_variable(
        density: Callable[[float], float],
        inverse: Callable[[float], float],
        *, eps: float = 1e-7,
) -> Callable[[float], float]:
    """Reparametrize a density to a new variable y, given the inverse map y -> theta.
    The Jacobian |d theta / d y| (the local stretch) is estimated by finite differences."""
    def transformed(y):
        theta = inverse(y)
        jacobian = np.abs((inverse(y + eps) - inverse(y - eps)) / (2.0 * eps))
        return density(theta) * jacobian
    return transformed


def ppv_posterior_from_theta_posterior(
        theta_posterior: Callable[[float], float],
        *, sensitivity: float, specificity: float,
) -> Callable[[float], float]:
    inverse = partial(theta_from_ppv, sensitivity=sensitivity, specificity=specificity)
    raw_ppv_posterior = change_of_variable(theta_posterior, inverse)
    # Re-normalize: PPV in [0, 1] maps to theta in [1 - spec, sens], so any theta-mass
    # outside that band (an impossible prevalence) is dropped and rescaled away.
    return normalize_posterior(raw_ppv_posterior)

## 6 · Assembling and visualizing the whole inference

Now we wire the pieces together. Pick the known test characteristics and the observed counts, choose a prior, then run the chain: prior → likelihood → unnormalized posterior → normalized posterior → PPV posterior. Every function below is one you wrote (or were handed) above.

In [ ]:
# --- what we know, and what we observed ---
sensitivity, specificity = 0.90, 0.85
num_positive_labels, num_negative_labels = 14, 26      # the classifier's verdicts on 40 samples

# --- the inference, one line per stage ---
# A Beta(8, 2) prior: as if we had already seen 8 positives and 2 negatives — a fairly
# strong hunch that the predicted-positive rate is high. Swap in Beta(1, 1) for a flat prior
# and watch the posterior collapse onto the likelihood.
chosen_prior = partial(prior, alpha=8.0, beta=2.0)
chosen_likelihood = partial(likelihood,
                            num_positive_labels=num_positive_labels,
                            num_negative_labels=num_negative_labels)

raw_posterior = get_unnormalized_posterior(chosen_prior, chosen_likelihood)
theta_posterior = normalize_posterior(raw_posterior)
ppv_posterior = ppv_posterior_from_theta_posterior(
    theta_posterior, sensitivity=sensitivity, specificity=specificity)

# A point summary: the posterior-mean PPV (integrate ppv x posterior over [0, 1]).
mean_ppv, _ = integrate.quad(lambda ppv: ppv * ppv_posterior(ppv), 0.0, 1.0)
print(f"posterior-mean PPV = {mean_ppv:.3f}")

### Plotting the components

In [ ]:
# Each plot takes a density as a function float -> float and samples it on a grid itself.

def plot_curve(f, title, *, color="#C45A26", domain=(0.0, 1.0), xlabel=r"$\theta$", n=400):
    """A single density over its domain — used to eyeball the prior or the likelihood."""
    x = np.linspace(domain[0], domain[1], n)
    y = f(x)
    fig, ax = plt.subplots(figsize=(7, 3.4))
    ax.plot(x, y, lw=2.4, color=color)
    ax.fill_between(x, y, color=color, alpha=0.10)
    ax.set_xlim(*domain); ax.set_ylim(bottom=0)
    ax.set_xlabel(xlabel); ax.set_title(title)
    fig.tight_layout(); plt.show()


def plot_three(prior_f, like_f, post_f, *, domain=(0.0, 1.0), n=400):
    """The prior - likelihood - posterior triptych. Prior and likelihood are rescaled to the
    posterior's height so the three shapes can be compared on one axis."""
    x = np.linspace(domain[0], domain[1], n)
    post = post_f(x)
    peak = np.max(post)
    to_peak = lambda v: v / np.max(v) * peak if np.max(v) > 0 else v
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.plot(x, to_peak(prior_f(x)), lw=2.2, color="#7C7563", label="prior (scaled)")
    ax.plot(x, to_peak(like_f(x)),  lw=2.2, color="#355E92", label="likelihood (scaled)")
    ax.plot(x, post, lw=3.0, color="#C45A26", label="posterior")
    ax.fill_between(x, post, color="#C45A26", alpha=0.10)
    ax.set_xlim(*domain); ax.set_ylim(bottom=0)
    ax.set_xlabel(r"$\theta$  (predicted-positive rate)"); ax.set_ylabel("density")
    ax.set_title("prior  -  likelihood  -  posterior")
    ax.legend(frameon=False)
    fig.tight_layout(); plt.show()


def plot_overlay(post_f, exact_f, *, domain=(0.0, 1.0), n=400):
    """Your numerical posterior against a closed-form reference — they should coincide."""
    x = np.linspace(domain[0], domain[1], n)
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.plot(x, post_f(x),  lw=4.0, color="#C45A26", alpha=0.55, label="numerical (your pipeline)")
    ax.plot(x, exact_f(x), lw=1.6, ls="--", color="#1A1A1A",
            label=r"analytic  Beta($\alpha+k,\ \beta+n-k$)")
    ax.set_xlim(*domain); ax.set_ylim(bottom=0)
    ax.set_xlabel(r"$\theta$"); ax.set_ylabel("density")
    ax.set_title("numerical posterior  vs  analytic Beta")
    ax.legend(frameon=False)
    fig.tight_layout(); plt.show()

In [ ]:
plot_three(chosen_prior, chosen_likelihood, theta_posterior)
plot_curve(ppv_posterior, "posterior for the PPV", color="#2E7D5B", xlabel="PPV")

## 7 · An easier way: conjugacy

The pipeline above works for *any* prior — that's its virtue. But we chose a Beta prior on purpose. The Beta family is the **conjugate prior** for the binomial likelihood: prior and posterior are both Beta, so the update has a closed form and needs no integration at all.

The intuition is, again, pseudo-counts. A Beta$(\alpha, \beta)$ prior behaves like having already seen $\alpha$ "positive" and $\beta$ "negative" pseudo-observations; its strength is $\alpha + \beta$, an equivalent number of samples. The data then add $k$ real positives and $n - k$ real negatives, and you simply add them up:

$$\text{Beta}(\alpha, \beta)\ \xrightarrow{\;k\text{ pos},\ n-k\text{ neg}\;}\ \text{Beta}(\alpha + k,\ \beta + n - k).$$

Let's compute that directly and overlay it on the numerical posterior. They should sit exactly on top of each other — a good check that our hand-rolled pipeline is correct.

In [ ]:
alpha = chosen_prior.keywords["alpha"]
beta = chosen_prior.keywords["beta"]
analytic_posterior = stats.beta(alpha + num_positive_labels,   # alpha + k
                                beta + num_negative_labels)     # beta + (n - k)

plot_overlay(theta_posterior, analytic_posterior.pdf)

grid = np.linspace(0.001, 0.999, 2000)
max_gap = np.max(np.abs(theta_posterior(grid) - analytic_posterior.pdf(grid)))
print(f"largest gap between numerical and analytic posterior: {max_gap:.2e}")

## Further reading

If you want to go deeper:

- **Kruschke, *Doing Bayesian Data Analysis*** — the gentlest on-ramp, with a practical cookbook you can lift code from.
- **McElreath, *Statistical Rethinking*** — rigorous but unusually readable, superb on the conceptual traps; the companion lecture course is free online.
- **Gelman et al., *Bayesian Data Analysis*** — *the* comprehensive modern reference. Demanding, but the place everything is done properly.